# Formation Data Augmentation

#### 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import glob

#### 2. Load and Explore the Dataset

In [2]:
# Load all cleaned match files
files = glob.glob("/home/falih/Mastercode/Bundesliga_2023_2024/formations/labled/labled_data_*.csv")
dfs = [pd.read_csv(f) for f in files]
df_matches = pd.concat(dfs, ignore_index=True)

In [ ]:
df_matches.value_counts("formation") // 10 # 10 = number of players per team

formation
4-2-3-1      342
remove       265
5-3-2        143
4-4-2        125
3-1-4-2      112
4-3-3        105
3-4-2-1       95
5-2-3         90
3-1-5-1       49
4-2-2-2       43
4-D-2         42
3-4-1-2       41
5-4-1         39
5-2-1-2       32
4-1-4-1       23
3-4-3         20
4-1-3-2       15
5-3-1-1       10
5-1-3-1        5
4-5-1          2
3-5-2          1
4-1-2-1-2      1
Name: count, dtype: int64

In [12]:
# Mask for the rows that should be mirrored
mask = ~df_matches['formation'].isin(['4-2-3-1', 'remove'])

# Create a copy of the relevant rows
df_mirrored = df_matches.loc[mask].copy()

# Apply mirroring (invert the y-axis)
df_mirrored['y_norm_mean'] = -df_mirrored['y_norm_mean']

# Assign new window numbers:
# Get the maximum window from the original data
max_window = df_matches['window'].max()

# Add this value + 1 to create continuous new window IDs
df_mirrored['window'] = df_mirrored['window'] + max_window + 1

# Combine original and mirrored data
df_aug = pd.concat([df_matches, df_mirrored], ignore_index=True)

# Check the distribution of formations
df_aug.value_counts("formation") // 10

formation
4-2-3-1      342
5-3-2        286
remove       265
4-4-2        250
3-1-4-2      224
4-3-3        210
3-4-2-1      190
5-2-3        180
3-1-5-1       98
4-2-2-2       86
4-D-2         84
3-4-1-2       82
5-4-1         78
5-2-1-2       64
4-1-4-1       46
3-4-3         40
4-1-3-2       30
5-3-1-1       20
5-1-3-1       10
4-5-1          4
3-5-2          2
4-1-2-1-2      2
Name: count, dtype: int64

#### 4. Team-based Global Window Grouping

In [ ]:
# Each group should represent one team frame (10 players)
group_cols = ["match_title", "game_section", "window", "team_code"]

# Create a sequential global ID for each unique window group
df_aug["window_global"] = df_aug.groupby(group_cols, sort=False).ngroup()

# Identify invalid windows (not exactly 10 players)
invalid_windows = df_aug["window_global"].value_counts()
invalid_windows = invalid_windows[invalid_windows != 10]

# Remove invalid windows
df_cleaned = df_aug[~df_aug["window_global"].isin(invalid_windows.index)].copy()

# Recalculate window_global after cleaning
df_cleaned["window_global"] = df_cleaned.groupby(group_cols, sort=False).ngroup()
df_cleaned.header()

,match_id,match_title,team_code,home_team_code,away_team_code,player_name,game_section,x_norm_mean,y_norm_mean,x_norm_std,y_norm_std,speed_mean,window,dominant_possession,window_start_sec,window_end_sec,window_duration_sec,dominant_possession_team,formation,window_global
0,DFL-MAT-J03YM4,1. FC Union Berlin:Sport-Club Freiburg,FCU,FCU,SCF,A. Schäfer,firstHalf,0.095523,-0.243839,0.420586,0.338717,12.068364,0,2,0.04,74.84,74.8,Out_of_possession,5-3-2,0
1,DFL-MAT-J03YM4,1. FC Union Berlin:Sport-Club Freiburg,FCU,FCU,SCF,B. Aaronson,firstHalf,1.403300,0.745084,0.347995,0.449502,13.801967,0,2,0.04,74.84,74.8,Out_of_possession,5-3-2,0
2,DFL-MAT-J03YM4,1. FC Union Berlin:Sport-Club Freiburg,FCU,FCU,SCF,C. Trimmel,firstHalf,0.037554,-1.677332,0.375631,0.166030,11.220161,0,2,0.04,74.84,74.8,Out_of_possession,5-3-2,0
3,DFL-MAT-J03YM4,1. FC Union Berlin:Sport-Club Freiburg,FCU,FCU,SCF,D. Doekhi,firstHalf,-0.563781,-0.887963,0.172994,0.207233,9.380860,0,2,0.04,74.84,74.8,Out_of_possession,5-3-2,0
4,DFL-MAT-J03YM4,1. FC Union Berlin:Sport-Club Freiburg,FCU,FCU,SCF,Diogo Leite,firstHalf,-0.372243,0.601369,0.277500,0.229887,11.020524,0,2,0.04,74.84,74.8,Out_of_possession,5-3-2,0


#### Save the augmented dataset

In [ ]:
df_cleaned.to_csv("/home/falih/Mastercode/Bundesliga_2023_2024/formations/labled/aug_data_window_global.csv", index=False)